In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.preprocessing import StandardScaler,MinMaxScaler,FunctionTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [5]:
TRAIN_PATH = Path.cwd() / "train"
TEST_PATH = Path.cwd() / "test"

In [6]:
# Load Training set 
X_train = pd.read_csv(TRAIN_PATH / "X_train.csv")
y_train = pd.read_csv(TRAIN_PATH / "y_train.csv")

# Load Testing set
X_test = pd.read_csv(TEST_PATH / "X_test.csv")
y_test = pd.read_csv(TEST_PATH / "y_test.csv")


# Will later be updated by a try-catch block

## Strategy for each column : 

#### 1. Pass-Through Features
* **`credit_score`**: Pass through without changes.

#### 2. Feature Engineering & Scaling
* **`person_emp_exp`** (Employment Experience):
    * **New Feature (`is_exp`)**: Indicator variable where `person_emp_exp == 0`.
    * **New Feature (`log_positive_exp`)**: Applies `np.log1p(x)` if `person_emp_exp > 0`, else `0`.
    * **Scaling**: Apply Min-Max Scaling to `log_positive_exp` values greater than 0.

#### 3. Binning & Discretisation
* **`person_age`**: Bin into four age groups:
    * `0–25`
    * `25–30`
    * `35–40`
    * `40+`
* **`loan_int_rate`**: Bin into two interest rate groups:
    * `< 11`
    * `> 11`

#### 4. Mathematical Transformations
* **`person_income`**: Apply `LogTransformer`.
* **`loan_amnt`**: Apply `SqrtTransformer`.
* **`loan_percent_income`**: Apply `SqrtTransformer`.


## Custom Classes and Transformers for Numeric Columns

In [ ]:
# Creating custom class for person_age
class CustomAgeBinning (BaseEstimator,TransformerMixin):
    """
    A custom class that slices the "person_age" column into intervals passes by user as a list.
    """
    
    def __init__(self,intervals):
        """
        intervals : Pass on the interval range as list ; for example : 
            range : 0-10,10-20,20-30,>30
            Pass this range as [0,10,20,30]
            The last interval will automatically take all the remaining values above it 
        dtype : list 
        """
        self.intervals = intervals

        # Whenever modifying data we store it into a new variable , also this new variable must be given a defensive level
        self._clean_intervals = sorted(list(intervals))

        # Also append infinity at the end to capture the end 
        if self._clean_intervals[-1] != np.inf : 
            self._clean_intervals.append(np.inf) 
    
    def fit(self,X,y=None):
        """
        Validates the dataslice passed and it's structure.
        """

## Custom Transformers and Strategy for Object Columns

In [ ]:
# We will make a column transformer for the encoders and create a custom Mapper function for our custom mapping cols

In [ ]:
X_train.select_dtypes(include="object").head()

,person_gender,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
35300,female,Associate,RENT,PERSONAL,Yes
9746,female,Master,RENT,MEDICAL,Yes
26766,female,High School,MORTGAGE,VENTURE,No
12944,female,High School,RENT,MEDICAL,No
466,male,Associate,RENT,HOMEIMPROVEMENT,No


In [ ]:
# To make sure the order for mapping cardinality values using OrdinalEncoder we will pass an explicit order to control behaviour
gender_order = ['female', 'male'] # Will map first value as 0 then next as 1 and so on...
file_order = ['No', 'Yes']

# Also the columns that we use OrdinalEncoder upon are : 
ord_cols = ["person_gender","previous_loan_defaults_on_file"]

In [ ]:
# Now OneHotEncoded cols will be as decided : 
onc_cols = ["person_home_ownership","loan_intent"]

In [ ]:
# Now handling our heirarchical col : person_education
heir_cols = ['person_education']

In [ ]:
# Now creating a custom mapper for person_education
def home_ownership_mapper(X):

    hierarchy_map = {
                            'High School': 1,
                            'Associate': 2,
                            'Bachelor': 3,
                            'Master': 4,
                            'Doctorate' : 5
    }

    # 1. Convert to a NumPy array so the format is always consistent
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()
    
    # 2. Create a vectorized version of your dictionary lookup.
    # The .get(val, 0) handles missing values or typos by defaulting to 0.
    vector_lookup = np.vectorize(lambda val: hierarchy_map.get(val, 0))
    
    # 3. Apply it. This outputs the exact same 2D shape that came in.
    return vector_lookup(X)

# Wrap it up safely
home_ownership_transformer = FunctionTransformer(home_ownership_mapper, validate=False)

# Chainging Steps into a  PIPELINE 

In [ ]:
# Column Transformer Sequential arranged 
col_transformer = ColumnTransformer(
        transformers = [
                            ("obj_enc_ord",OrdinalEncoder(
                                                                categories=[gender_order,file_order],
                                                                handle_unknown='use_encoded_value',
                                                                unknown_value=-1
                            ),ord_cols),
                            ("obj_enc_onc",OneHotEncoder(drop='first',handle_unknown='error'),onc_cols),
                            ("obj_enc_heir",home_ownership_transformer,heir_cols)
        ],
        remainder='drop'
)

In [ ]:
# Create the final pipeline
lor_pipeline = Pipeline(steps=[
    ('preprocessor', col_transformer),
    ('lor', LogisticRegression(max_iter=1000))
])

# Fit everything 
lor_pipeline.fit(X_train, y_train)